# ALARB week 11: do retrieved articles help verdict prediction?

ALARB's authors tried retrieving statute articles from case facts, found it
"extremely poor", and gave the model the court's own articles instead (their task 2).
A real assistant never has those. This notebook tests the realistic setting: Qwen3-8B
predicts verdicts for the same 20 development-test cases under four conditions.

| Condition | Articles in the prompt |
|---|---|
| Facts only (paper task 1) | none |
| Most-cited articles (control) | the 5 most-cited training articles, the same for every case |
| Retrieved articles (our method) | top 5 from a retriever indexed on training cases only |
| Court-cited articles (paper task 2) | the articles the court actually cited (ceiling) |

The three article conditions use the paper's Figure 10 prompt word for word.
Every verdict is graded with the paper's Appendix B.1 judge prompt.

**Before running:** Runtime > Change runtime type > **T4 GPU**. No paid plan or API key
is needed. Nothing is saved to Google Drive; the final cell prints the results.
The final validation holdout is never downloaded.


## 1 Check the GPU


In [ ]:
import sys, torch
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then run again.")
print("GPU:", torch.cuda.get_device_name(0), "|", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GiB")


## 2 Install pinned libraries


In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1",
                "accelerate==1.10.1", "bitsandbytes==0.48.1", "datasets==4.0.0"], check=True)


## 3 Load our code from GitHub


In [ ]:
from pathlib import Path
import os
PROJECT = Path("/content/alarb-project")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "week11-retrieval-novelty",
                    "https://github.com/RAbumansi/week1-alarb-reproduction.git", str(PROJECT)], check=True)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Code commit:", COMMIT)


## 4 Freeze the settings
Sampling follows Qwen's guidance for thinking mode; the paper does not state its decoding settings. Do not change the seed after seeing results.


In [ ]:
import json
from dataclasses import asdict
from alarb import colab_week11 as w
from alarb.colab_demo import load_development, prepare_development
CONFIG = w.Week11Config(cases=20, seed=42, batch_size=10)
RESULTS = Path("/content/week11-results")
print(json.dumps(asdict(CONFIG), indent=2))


## 5 Prepare the cases and the four prompts
Only the publisher's training file is downloaded (its SHA-256 is checked), and the course split is rebuilt from it. The retriever's index uses the 9,339 training cases; the 20 test cases' own citations are read only to build the ceiling condition and to measure recall.


In [ ]:
source = load_development(Path("/content/alarb-source-cache"))
training, testing, _, _ = prepare_development(source, CONFIG.split_seed)
cases = w.select_cases(testing, CONFIG.cases, CONFIG.seed)
conditions, preparation = w.build_conditions(training, cases)
preparation.update({"training_cases": len(training), "development_test_cases": len(testing),
                    "selected_case_ids": [c.case_id for c in cases], "code_commit": COMMIT})
print("Training:", len(training), "| Development test:", len(testing), "| Selected:", len(cases))
print("Retrieval recall@5 on these cases:", preparation["recall@5_on_selected_cases"])
print("Final validation: not downloaded")


## 6 Load Qwen3-8B on the GPU
The weights download to this Colab machine (several minutes), not to your computer.


In [ ]:
if "backend" not in globals():
    backend = w.BatchedBackend(CONFIG)
print(json.dumps(backend.metadata, indent=2))


## 7 Generate and grade each condition
Each condition is generated, graded with the paper's Appendix B.1 judge prompt, and its results printed before the next starts, so a dropped session loses at most one condition. To skip conditions finished in an earlier session, paste their printed output into `EARLIER` as a string.


In [ ]:
EARLIER = ""
results = w.parse_printed(EARLIER)
for (task, retriever), cid in zip(w.CONDITIONS, w.CONDITION_IDS):
    if cid in results:
        print(f"  {cid}: recovered from an earlier session")
        continue
    directory = RESULTS / "runs" / cid
    w.run_condition(conditions[cid], backend, directory, CONFIG, cid, retriever)
    w.judge_condition(directory, backend, CONFIG, cid)
    results[cid] = w.compact(directory)
    w.print_condition(cid, results[cid])


## 8 Results
Case hashes, labels and counts only; no case text or names.


In [ ]:
import pandas as pd
summary = w.summarise(results, CONFIG, preparation, backend.metadata, w.judge_label(CONFIG))
display(pd.DataFrame([{
    "Condition": r["condition"], "Cases": r["cases"],
    "Correct %": round(100 * r["labels"]["CORRECT"]), "Partially %": round(100 * r["labels"]["PARTIALLY CORRECT"]),
    "Incorrect %": round(100 * (r["labels"]["INCORRECT"] + r["labels"]["UNPARSEABLE"])),
    "Correct or partial %": round(100 * r["correct_or_partial"]),
    "Outcome agreement %": None if r["outcome_agreement"]["share"] is None else round(100 * r["outcome_agreement"]["share"]),
    "Better / worse vs facts": "" if "vs_baseline" not in r else f'{r["vs_baseline"]["better"]} / {r["vs_baseline"]["worse"]} (p={r["vs_baseline"]["sign_test_p"]})',
    "Truncated": r["truncated"],
} for r in summary["comparison"]["rows"]]))
w.print_result(summary)
